# Constrained generation (OpenRouter)

Same prompt, two runs:

| | Baseline | Constrained |
|---|---|---|
| messages | user only | system + strict user |
| stop | none | cut common postambles |
| expect | prose + code + extras | mostly just the code |

In [1]:
import os
from pathlib import Path

import requests
from dotenv import load_dotenv

load_dotenv(Path("/Users/garvitkhurana/Projects/llm-api-compare") / ".env")

API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
assert API_KEY, "Set OPENROUTER_API_KEY in .env"

URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"

PROMPT = "Python code to give a sqrt of pi to 6 decimal places."
SYSTEM = "Be terse. Output only what was asked. No greetings or extras."
STOP = ["**Output", "Output:", "Here are", "Sure,"]

print("ok")

ok


/Users/garvitkhurana/Projects/llm-api-compare/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
def chat(messages, stop=None, temperature=0.2, max_tokens=600):
    payload = {
        "model": MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    if stop:
        payload["stop"] = stop

    r = requests.post(
        URL,
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
        json=payload,
        timeout=120,
    )
    data = r.json()
    if not r.ok or "error" in data or not data.get("choices"):
        raise RuntimeError(data.get("error") or data or r.text)

    choice = data["choices"][0]
    text = (choice.get("message") or {}).get("content") or ""
    return text, choice.get("finish_reason"), data.get("usage")

In [3]:
msgs_baseline = [{"role": "user", "content": PROMPT}]

msgs_constrained = [
    {"role": "system", "content": SYSTEM},
    {
        "role": "user",
        "content": PROMPT + "\nReply with only the Python code. No markdown fences, no explanation.",
    },
]

base_text, base_fr, base_usage = chat(msgs_baseline, temperature=0.7, max_tokens=800)
cons_text, cons_fr, cons_usage = chat(msgs_constrained, stop=STOP, temperature=0.2, max_tokens=400)

print("=" * 60)
print("BASELINE")
print("finish_reason:", base_fr, "| usage:", base_usage)
print("-" * 60)
print(base_text)

print("\n" + "=" * 60)
print("CONSTRAINED")
print("finish_reason:", cons_fr, "| usage:", cons_usage)
print("-" * 60)
print(cons_text)

BASELINE
finish_reason: stop | usage: {'prompt_tokens': 30, 'completion_tokens': 76, 'total_tokens': 106, 'cost': 0, 'is_byok': False, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}, 'completion_tokens_details': {'reasoning_tokens': 21, 'image_tokens': 0, 'audio_tokens': 0}}
------------------------------------------------------------
```python
import math

# Calculate square root of pi
sqrt_pi = math.sqrt(math.pi)

# Print formatted to 6 decimal places
print(f"{sqrt_pi:.6f}")
```

**Output:**
```
1.772454
```

CONSTRAINED
finish_reason: stop | usage: {'prompt_tokens': 60, 'completion_tokens': 48, 'total_tokens': 108, 'cost': 0, 'is_byok': False, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0,

In [4]:
from IPython.display import Markdown, display

def _tok(usage, key):
    return (usage or {}).get(key)

rows = [
    ("| metric | baseline | constrained |"),
    ("|---|---:|---:|"),
    (f"| finish_reason | {base_fr} | {cons_fr} |"),
    (f"| prompt_tokens | {_tok(base_usage, 'prompt_tokens')} | {_tok(cons_usage, 'prompt_tokens')} |"),
    (f"| completion_tokens | {_tok(base_usage, 'completion_tokens')} | {_tok(cons_usage, 'completion_tokens')} |"),
    (f"| total_tokens | {_tok(base_usage, 'total_tokens')} | {_tok(cons_usage, 'total_tokens')} |"),
    (f"| chars | {len(base_text)} | {len(cons_text)} |"),
    (f"| preview | `{base_text[:80].replace(chr(10), ' ')}` | `{cons_text[:80].replace(chr(10), ' ')}` |"),
]
display(Markdown("\n".join(rows)))

| metric | baseline | constrained |
|---|---:|---:|
| finish_reason | stop | stop |
| prompt_tokens | 30 | 60 |
| completion_tokens | 76 | 48 |
| total_tokens | 106 | 108 |
| chars | 178 | 46 |
| preview | ````python import math  # Calculate square root of pi sqrt_pi = math.sqrt(math.pi` | `import math print(f"{math.sqrt(math.pi):.6f}")` |